# 🎬 AI Clipping Bot - Whop Campaign Automation
**Powered by Google Gemini 2.5 Flash + FFmpeg | 100% Gratuit**

---
### Comment utiliser ce notebook :
1. Remplissez vos clés dans la **Cellule 1 (Config)**
2. Cliquez sur **Runtime → Run All** (ou Ctrl+F9)
3. Vos clips seront dans votre **Google Drive** !

> ⚠️ Gardez l'onglet Colab ouvert pendant le traitement.

In [ ]:
# ============================================================
# CELLULE 1 - CONFIGURATION (modifiez uniquement ici)
# ============================================================

GEMINI_API_KEY = 'COLLEZ_VOTRE_CLE_GEMINI_ICI'
GITHUB_TOKEN    = 'COLLEZ_VOTRE_TOKEN_GITHUB_ICI'
GITHUB_USER     = 'mrdarkness5298-blip'
GITHUB_REPO     = 'ai-clipping-bot'

BRIEF_URL = 'https://docs.google.com/document/d/1modTZq5jJ5WkYZ1TmitNcV7MfHxTTRyGfKsFLdqCqUM/edit'

CLIPS_TO_GENERATE = 20
MIN_CLIP_DURATION = 10
MAX_CLIP_DURATION = 55
DRIVE_OUTPUT_FOLDER = 'AI_Clips_Output'

print('Configuration OK')

In [ ]:
# ============================================================
# CELLULE 2 - INSTALLATION
# ============================================================
print('Installation des dependances...')
!apt-get install -y ffmpeg -q
!pip install -U -q google-genai google-generativeai gdown

import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print(f'FFmpeg: {result.stdout.split(chr(10))[0]}')
print('Installation complete !')

In [ ]:
# ============================================================
# CELLULE 3 - CONNEXION GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

OUTPUT_DIR   = Path(f'/content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}')
DOWNLOAD_DIR = Path('/content/downloads')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(exist_ok=True)

print(f'Drive monte ! Clips -> {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELLULE 4 - LECTURE DU BRIEF GOOGLE DOCS
# ============================================================
import re, requests
from html.parser import HTMLParser

def read_brief(doc_url):
    doc_id = re.search(r'/document/d/([a-zA-Z0-9_-]+)', doc_url).group(1)
    html = requests.get(
        f'https://docs.google.com/document/d/{doc_id}/export?format=html',
        timeout=30
    ).text

    class Parser(HTMLParser):
        def __init__(self):
            super().__init__()
            self.text, self.links = [], []
        def handle_starttag(self, tag, attrs):
            if tag == 'a':
                href = dict(attrs).get('href', '')
                if 'google.com/url?q=' in href:
                    from urllib.parse import unquote
                    m = re.search(r'q=([^&]+)', href)
                    if m: href = unquote(m.group(1))
                if href: self.links.append(href)
        def handle_data(self, d):
            self.text.append(d)

    p = Parser()
    p.feed(html)
    text = ' '.join(p.text)
    drive_links = []
    for href in p.links:
        if 'drive.google.com' in href and href not in drive_links:
            drive_links.append(href.rstrip('.,;)'))
    raw = re.findall(r'https://drive\.google\.com/[^\s"<>&]+', html)
    for l in raw:
        c = l.rstrip('.,;)')
        if c not in drive_links:
            drive_links.append(c)
    print(f'Brief lu : {len(drive_links)} lien(s) Drive')
    return {'text': text, 'drive_links': drive_links}

brief = read_brief(BRIEF_URL)
print(f'Apercu: {brief["text"][:200]}')

In [ ]:
# ============================================================
# CELLULE 5 - TELECHARGEMENT DES VIDEOS DEPUIS DRIVE
# ============================================================
import time, json, gdown

def get_folder_id(url):
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
    return m.group(1) if m else None

VIDEO_EXT    = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
seen_folders = set()

for url in brief['drive_links']:
    folder_id = get_folder_id(url)
    if folder_id and folder_id not in seen_folders:
        seen_folders.add(folder_id)
        print(f'\nDossier : {folder_id}')
        try:
            files = gdown.download_folder(
                url=f'https://drive.google.com/drive/folders/{folder_id}',
                output=str(DOWNLOAD_DIR) + '/',
                quiet=False,
                use_cookies=False
            ) or []
            print(f'  {len(files)} fichier(s) telecharges')
        except Exception as e:
            print(f'  Erreur: {e}')

video_files = sorted([
    p for p in DOWNLOAD_DIR.rglob('*')
    if p.suffix.lower() in VIDEO_EXT and p.stat().st_size > 500_000
])
print(f'\n{len(video_files)} video(s) prete(s):')
for v in video_files:
    print(f'  {v.name} ({v.stat().st_size/(1024*1024):.1f} MB)')

In [ ]:
# ============================================================
# CELLULE 6 - ANALYSE IA AVEC GEMINI 1.5 FLASH
# ============================================================
from google import genai
from google.genai import types
import subprocess, json, time

client = genai.Client(api_key=GEMINI_API_KEY)

SYSTEM_PROMPT = """
Tu es un expert en montage video viral pour TikTok et Instagram Reels.
Analyse la video et selectionne les meilleurs segments selon le brief.
REGLE : reponds UNIQUEMENT en JSON valide, sans texte autour.
FORMAT:
{"clips": [{"id": 1, "start": 5.0, "end": 30.0, "duration": 25.0, "score": 90, "reason": "Moment fort"}],
 "video_summary": "Description courte"}
"""

def get_duration(path):
    r = subprocess.run(['ffprobe', '-v', 'quiet', '-print_format', 'json',
                        '-show_format', str(path)], capture_output=True, text=True)
    try: return float(json.loads(r.stdout)['format']['duration'])
    except: return 60.0

def analyze_video(video_path, n_clips, min_dur, max_dur):
    print(f'  Upload Gemini...')
    vf = client.files.upload(file=str(video_path), config={'display_name': video_path.name})
    while vf.state == 'PROCESSING':
        time.sleep(3)
        vf = client.files.get(name=vf.name)
    if vf.state == 'FAILED':
        raise RuntimeError('Gemini ne peut pas traiter cette video')
        
    prompt = f"""
Tu es un monteur video expert. Voici le brief de la campagne :
{brief['text'][:2000]}

MISSION OBLIGATOIRE :
Trouve EXACTEMENT {n_clips} moments pertinents dans la video fournie qui correspondent au brief.
Chaque moment doit durer entre {min_dur} et {max_dur} secondes.
Tu DOIS repondre UNIQUEMENT avec un JSON valide. N'ajoute AUCUN texte avant ou apres le JSON.

FORMAT ATTENDU :
{{
  "clips": [
    {{"id": 1, "start": 10.0, "end": 25.0, "duration": 15.0, "score": 95, "reason": "Raison courte", "caption": "Texte d'accroche viral a afficher sur la video (max 10 mots)", "description": "Description pour le post TikTok/Reels avec hashtags, tres engageante et conforme au brief"}}
  ]
}}
"""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model='gemini-1.5-flash',
                contents=[vf, prompt],
                config=types.GenerateContentConfig(temperature=0.2, response_mime_type="application/json")
            )
            raw = resp.text.strip()
            break
        except Exception as e:
            if attempt < max_retries - 1:
                print(f'    Surcharge/Quota API (Attente 30s avant reessai...)')
                time.sleep(30)
            else:
                raise RuntimeError(f'Erreur API: {str(e)}')

    try: client.files.delete(name=vf.name)
    except: pass
    
    raw = raw.replace('```json', '').replace('```', '').strip()
    try:
        data = json.loads(raw)
        clips = data.get('clips', [])
        if not clips: raise RuntimeError(f'Reponse vide de Gemini: {raw[:200]}')
        return clips
    except json.JSONDecodeError:
        raise RuntimeError(f'JSON invalide: {raw[:200]}')

durations = {str(v): get_duration(v) for v in video_files}
total_dur = sum(durations.values())
analysis_results = {}

for i, video_path in enumerate(video_files):
    if i > 0:
        time.sleep(10) # Pause pour eviter la limite de la version gratuite (15 requetes/minute)
    dur = durations[str(video_path)]
    n = max(1, round((dur / total_dur) * CLIPS_TO_GENERATE)) if total_dur > 0 else 1
    n = min(n, max(1, int(dur // MIN_CLIP_DURATION)))
    print(f'\n[{i+1}/{len(video_files)}] {video_path.name} -> {n} clips ({dur:.0f}s)')
    try:
        clips = analyze_video(video_path, n, MIN_CLIP_DURATION, MAX_CLIP_DURATION)
        analysis_results[str(video_path)] = {"clips": clips, "error": None}
        print(f'  OK: {len(clips)} clips trouves')
    except Exception as e:
        print(f'  ERREUR: {e}')
        analysis_results[str(video_path)] = {"clips": [], "error": str(e)}

total = sum(len(v['clips']) for v in analysis_results.values())
print(f'\nTotal: {total} clips identifies par Gemini')


In [ ]:
# ============================================================
# CELLULE 7 - MONTAGE VIDEO FFMPEG (9:16 vertical)
# ============================================================
from datetime import datetime

campaign_name = 'soul_tied'
created_clips = []

for video_path_str, data in analysis_results.items():
    clips = data.get('clips', [])
    if not clips: continue
    video_path = Path(video_path_str)
    dur_total  = durations[video_path_str]
    for clip in clips:
        start    = float(clip.get('start', 0))
        end      = min(float(clip.get('end', start+30)), dur_total)
        cid      = clip.get('id', len(created_clips)+1)
        ts       = datetime.now().strftime('%H%M%S')
        out_name = f'{campaign_name}_clip_{cid:02d}_{ts}.mp4'
        out_path = OUTPUT_DIR / out_name
        cmd = [
            'ffmpeg', '-y', '-ss', str(start), '-i', str(video_path),
            '-t', str(end - start),
            '-vf', 'scale=-2:1920,crop=1080:1920',
            '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
            '-c:a', 'aac', '-b:a', '128k', '-movflags', '+faststart',
            str(out_path)
        ]
        result = subprocess.run(cmd, capture_output=True, timeout=300)
        if result.returncode == 0 and out_path.exists():
            size_mb = out_path.stat().st_size / (1024*1024)
            print(f'  OK: {out_name} ({size_mb:.1f} MB)')
            
            # Sauvegarder la caption et la description dans un fichier texte
            txt_path = out_path.with_suffix('.txt')
            with open(txt_path, 'w', encoding='utf-8') as txt_file:
                txt_file.write(f"--- CAPTION (Texte sur la video) ---\n{clip.get('caption', '')}\n\n")
                txt_file.write(f"--- DESCRIPTION (Pour le post) ---\n{clip.get('description', '')}\n")
            
            created_clips.append((out_path, clip.get('caption', ''), clip.get('description', '')))
        else:
            print(f'  ERREUR clip {cid}: {result.stderr[-150:]}')

print(f'\n{len(created_clips)} clips crees dans Google Drive !')

In [ ]:
# ============================================================
# CELLULE 8 - RAPPORT FINAL + ENVOI SUR GITHUB
# ============================================================
import base64
from datetime import datetime

now = datetime.now().strftime('%Y-%m-%d %H:%M')

# Construire le rapport
lines = [
    f'# Rapport - AI Clipping Bot',
    f'**Date :** {now}',
    f'**Brief :** {BRIEF_URL}',
    '',
    '## Résultats',
    f'- Videos analysees : **{len(video_files)}**',
    f'- Clips generes    : **{len(created_clips)}**',
    f'- Dossier Drive    : `{DRIVE_OUTPUT_FOLDER}`',
    '',
    '## Clips créés',
]
for p, cap, desc in created_clips:
    mb = p.stat().st_size / (1024*1024)
    lines.append(f'- `{p.name}` ({mb:.1f} MB)')
    lines.append(f'  > **Caption:** {cap}')
    lines.append(f'  > **Description:** {desc}')
    lines.append('')

lines += [
    '',
    '## Analyse par video',
]
for vp_str, data in analysis_results.items():
    vname = Path(vp_str).name
    clips = data['clips']
    err = data['error']
    if err:
        lines.append(f'\n### ❌ {vname} (ERREUR)\n> {err}')
    else:
        lines.append(f'\n### ✅ {vname} ({len(clips)} clips)')
        for c in clips:
            lines.append(f'- Clip {c.get("id")}: {c.get("start",0):.1f}s -> {c.get("end",0):.1f}s | Score: {c.get("score","?")} | {c.get("reason","")}')

report_md = '\n'.join(lines)
print(report_md[:500])

# Envoyer sur GitHub
gh_headers = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}
report_filename = f'rapports/rapport_{datetime.now().strftime("%Y%m%d_%H%M")}.md'
content_b64 = base64.b64encode(report_md.encode()).decode()

r = requests.put(
    f'https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}/contents/{report_filename}',
    headers=gh_headers,
    json={'message': f'Rapport {now}', 'content': content_b64}
)

if r.status_code in (200, 201):
    print(f'\n✅ Rapport envoye sur GitHub !')
    print(f'   https://github.com/{GITHUB_USER}/{GITHUB_REPO}/tree/main/rapports')
else:
    print(f'\n⚠️ GitHub non disponible ({r.status_code}) - rapport affiche ci-dessus')

print('\n' + '='*50)
print(f'  TERMINE ! {len(created_clips)} clips dans votre Drive.')
print('='*50)